In [15]:
import pandas as p
from glob import glob
from scipy.stats import hypergeom

In [16]:
slim = Annotation.objects.get(name='GO SLIM').get_annotations()
pc = {}
for row in p.read_csv('/home/matej/todo/mojca/darj/protein_complex_standard.csv').itertuples(index=False):
    pc[(row[1], row[1])] = row[2].split(';')
background = set(p.read_csv('/home/matej/todo/mojca/darj/array_background_v5.txt').ORF)

In [17]:
def dostuff(query, annot):
    m = len(background)  # M
    nn = len(query)  # N

    vals = []
    max_hits = 0
    for term, term_genes in annot.items():
        category = background.intersection(term_genes)
        n = len(category)
        hits = query.intersection(category)
        x = len(hits)
        max_hits = max(max_hits, x)

        if 0 in (n, x):
            # either no genes in this term or no hits in this term
            continue

        vals.append(term + (hypergeom.sf(x - 1, m, n, nn), x, n, nn, m, ','.join(hits)))

    df = p.DataFrame(vals, columns=['go_id', 'go_name', 'pval', 'hits_in_term', 'term_size', 'all_hits',
                                    'all_background', 'genes'])
    df = df.sort_values('pval')
    df.loc[:, 'bonferroni'] = df.pval * df.shape[0]
    df.loc[:, 'bonferroni'] = df.loc[:, 'bonferroni'].clip(upper=1)
    df.loc[:, 'fold_enrichment'] = (df.hits_in_term / df.all_hits) / (df.term_size / df.all_background)

    df = df.loc[df.hits_in_term > 0]
    

    return df

In [18]:
for f in glob('/home/matej/todo/mojca/darj/v2_v5_biounknown/*txt'):
    orf = f.split('/')[-1][:-4]
    try:
        query = set(p.read_csv(f, header=None).iloc[:,0].tolist())
    except:
        continue
    
    dfpc = dostuff(query, pc)
    dfslim = dostuff(query, slim)
    dfpc.to_csv('/home/matej/todo/mojca/darj/v2_v5_biounknown_enrich/%s_pc.csv' % orf)
    dfslim.to_csv('/home/matej/todo/mojca/darj/v2_v5_biounknown_enrich/%s_slim.csv' % orf)